# 01 — Train Wgan / 生成对抗网络

**Chapter 16 — File 1 of 2 / 第16章 — 第1个文件（共2个）**

---

## Summary / 总结

This script demonstrates **example of a wgan for generating handwritten digits**.

本脚本演示 **example of a wgan for generating handwritten digits**。

---
## Step 1 — example of a wgan for generating handwritten digits

In [ ]:
from numpy import expand_dims
from numpy import mean
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.mnist import load_data
from keras import backend
from keras.optimizers import RMSprop
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import BatchNormalization
from keras.initializers import RandomNormal
from keras.constraints import Constraint
from matplotlib import pyplot

---
## Step 2 — clip model weights to a given hypercube

In [ ]:
class ClipConstraint(Constraint):

---
## Step 3 — set clip value when initialized

In [ ]:
def __init__(self, clip_value):
		self.clip_value = clip_value

---
## Step 4 — clip model weights to hypercube

In [ ]:
def __call__(self, weights):
		return backend.clip(weights, -self.clip_value, self.clip_value)

---
## Step 5 — get the config

In [ ]:
def get_config(self):
		return {'clip_value': self.clip_value}

---
## Step 6 — calculate wasserstein loss

In [ ]:
def wasserstein_loss(y_true, y_pred):
	return backend.mean(y_true * y_pred)

---
## Step 7 — define the standalone critic model

In [ ]:
def define_critic(in_shape=(28,28,1)):

---
## Step 8 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 9 — weight constraint

In [ ]:
const = ClipConstraint(0.01)

---
## Step 10 — define model

In [ ]:
model = Sequential()

---
## Step 11 — downsample to 14x14

In [ ]:
model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init, kernel_constraint=const, input_shape=in_shape))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))

---
## Step 12 — downsample to 7x7

In [ ]:
model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init, kernel_constraint=const))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))

---
## Step 13 — scoring, linear activation

In [ ]:
model.add(Flatten())
	model.add(Dense(1))

---
## Step 14 — compile model

In [ ]:
opt = RMSprop(lr=0.00005)
	model.compile(loss=wasserstein_loss, optimizer=opt)
	return model

---
## Step 15 — define the standalone generator model

In [ ]:
def define_generator(latent_dim):

---
## Step 16 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 17 — define model

In [ ]:
model = Sequential()

---
## Step 18 — foundation for 7x7 image

In [ ]:
n_nodes = 128 * 7 * 7
	model.add(Dense(n_nodes, kernel_initializer=init, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((7, 7, 128)))

---
## Step 19 — upsample to 14x14

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))

---
## Step 20 — upsample to 28x28

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))

---
## Step 21 — output 28x28x1

In [ ]:
model.add(Conv2D(1, (7,7), activation='tanh', padding='same', kernel_initializer=init))
	return model

---
## Step 22 — define the combined generator and critic model, for updating the generator

In [ ]:
def define_gan(generator, critic):

---
## Step 23 — make weights in the critic not trainable

In [ ]:
for layer in critic.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False

---
## Step 24 — connect them

In [ ]:
model = Sequential()

---
## Step 25 — add generator

In [ ]:
model.add(generator)

---
## Step 26 — add the critic

In [ ]:
model.add(critic)

---
## Step 27 — compile model

In [ ]:
opt = RMSprop(lr=0.00005)
	model.compile(loss=wasserstein_loss, optimizer=opt)
	return model

---
## Step 28 — load images

In [ ]:
def load_real_samples():

---
## Step 29 — load dataset

In [ ]:
(trainX, trainy), (_, _) = load_data()

---
## Step 30 — select all of the examples for a given class

In [ ]:
selected_ix = trainy == 7
	X = trainX[selected_ix]

---
## Step 31 — expand to 3d, e.g. add channels

In [ ]:
X = expand_dims(X, axis=-1)

---
## Step 32 — convert from ints to floats

In [ ]:
X = X.astype('float32')

---
## Step 33 — scale from [0,255] to [-1,1]

In [ ]:
X = (X - 127.5) / 127.5
	return X

---
## Step 34 — select real samples

In [ ]:
def generate_real_samples(dataset, n_samples):

---
## Step 35 — choose random instances

In [ ]:
ix = randint(0, dataset.shape[0], n_samples)

---
## Step 36 — select images

In [ ]:
X = dataset[ix]

---
## Step 37 — generate class labels, -1 for 'real'

In [ ]:
y = -ones((n_samples, 1))
	return X, y

---
## Step 38 — generate points in latent space as input for the generator

In [ ]:
def generate_latent_points(latent_dim, n_samples):

---
## Step 39 — generate points in the latent space

In [ ]:
x_input = randn(latent_dim * n_samples)

---
## Step 40 — reshape into a batch of inputs for the network

In [ ]:
x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

---
## Step 41 — use the generator to generate n fake examples, with class labels

In [ ]:
def generate_fake_samples(generator, latent_dim, n_samples):

---
## Step 42 — generate points in latent space

In [ ]:
x_input = generate_latent_points(latent_dim, n_samples)

---
## Step 43 — predict outputs

In [ ]:
X = generator.predict(x_input)

---
## Step 44 — create class labels with 1.0 for 'fake'

In [ ]:
y = ones((n_samples, 1))
	return X, y

---
## Step 45 — generate samples and save as a plot and save the model

In [ ]:
def summarize_performance(step, g_model, latent_dim, n_samples=100):

---
## Step 46 — prepare fake examples

In [ ]:
X, _ = generate_fake_samples(g_model, latent_dim, n_samples)

---
## Step 47 — scale from [-1,1] to [0,1]

In [ ]:
X = (X + 1) / 2.0

---
## Step 48 — plot images

In [ ]:
for i in range(10 * 10):

---
## Step 49 — define subplot

In [ ]:
pyplot.subplot(10, 10, 1 + i)

---
## Step 50 — turn off axis

In [ ]:
pyplot.axis('off')

---
## Step 51 — plot raw pixel data

In [ ]:
pyplot.imshow(X[i, :, :, 0], cmap='gray_r')

---
## Step 52 — save plot to file

In [ ]:
filename1 = 'generated_plot_%04d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()

---
## Step 53 — save the generator model

In [ ]:
filename2 = 'model_%04d.h5' % (step+1)
	g_model.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

---
## Step 54 — create a line plot of loss for the gan and save to file

In [ ]:
def plot_history(d1_hist, d2_hist, g_hist):

---
## Step 55 — plot history

In [ ]:
pyplot.plot(d1_hist, label='crit_real')
	pyplot.plot(d2_hist, label='crit_fake')
	pyplot.plot(g_hist, label='gen')
	pyplot.legend()
	pyplot.savefig('plot_line_plot_loss.png')
	pyplot.close()

---
## Step 56 — train the generator and critic

In [ ]:
def train(g_model, c_model, gan_model, dataset, latent_dim, n_epochs=10, n_batch=64, n_critic=5):

---
## Step 57 — calculate the number of batches per training epoch

In [ ]:
bat_per_epo = int(dataset.shape[0] / n_batch)

---
## Step 58 — calculate the number of training iterations

In [ ]:
n_steps = bat_per_epo * n_epochs

---
## Step 59 — calculate the size of half a batch of samples

In [ ]:
half_batch = int(n_batch / 2)

---
## Step 60 — lists for keeping track of loss

In [ ]:
c1_hist, c2_hist, g_hist = list(), list(), list()

---
## Step 61 — manually enumerate epochs

In [ ]:
for i in range(n_steps):

---
## Step 62 — update the critic more than the generator

In [ ]:
c1_tmp, c2_tmp = list(), list()
		for _ in range(n_critic):

---
## Step 63 — get randomly selected 'real' samples

In [ ]:
X_real, y_real = generate_real_samples(dataset, half_batch)

---
## Step 64 — update critic model weights

In [ ]:
c_loss1 = c_model.train_on_batch(X_real, y_real)
			c1_tmp.append(c_loss1)

---
## Step 65 — generate 'fake' examples

In [ ]:
X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)

---
## Step 66 — update critic model weights

In [ ]:
c_loss2 = c_model.train_on_batch(X_fake, y_fake)
			c2_tmp.append(c_loss2)

---
## Step 67 — store critic loss

In [ ]:
c1_hist.append(mean(c1_tmp))
		c2_hist.append(mean(c2_tmp))

---
## Step 68 — prepare points in latent space as input for the generator

In [ ]:
X_gan = generate_latent_points(latent_dim, n_batch)

---
## Step 69 — create inverted labels for the fake samples

In [ ]:
y_gan = -ones((n_batch, 1))

---
## Step 70 — update the generator via the critic's error

In [ ]:
g_loss = gan_model.train_on_batch(X_gan, y_gan)
		g_hist.append(g_loss)

---
## Step 71 — summarize loss on this batch

In [ ]:
print('>%d, c1=%.3f, c2=%.3f g=%.3f' % (i+1, c1_hist[-1], c2_hist[-1], g_loss))

---
## Step 72 — evaluate the model performance every 'epoch'

In [ ]:
if (i+1) % bat_per_epo == 0:
			summarize_performance(i, g_model, latent_dim)

---
## Step 73 — line plots of loss

In [ ]:
plot_history(c1_hist, c2_hist, g_hist)

---
## Step 74 — size of the latent space

In [ ]:
latent_dim = 50

---
## Step 75 — create the critic

In [ ]:
critic = define_critic()

---
## Step 76 — create the generator

In [ ]:
generator = define_generator(latent_dim)

---
## Step 77 — create the gan

In [ ]:
gan_model = define_gan(generator, critic)

---
## Step 78 — load image data

In [ ]:
dataset = load_real_samples()
print(dataset.shape)

---
## Step 79 — train model

In [ ]:
train(generator, critic, gan_model, dataset, latent_dim)

---
## Learning Notes / 学习笔记

- **概念**: example of a wgan for generating handwritten digits 是机器学习中的常用技术。  
  *example of a wgan for generating handwritten digits is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Wgan / 生成对抗网络
# Complete Code / 完整代码
# ===============================

# example of a wgan for generating handwritten digits
from numpy import expand_dims
from numpy import mean
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.mnist import load_data
from keras import backend
from keras.optimizers import RMSprop
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import BatchNormalization
from keras.initializers import RandomNormal
from keras.constraints import Constraint
from matplotlib import pyplot

# clip model weights to a given hypercube
class ClipConstraint(Constraint):
	# set clip value when initialized
	def __init__(self, clip_value):
		self.clip_value = clip_value

	# clip model weights to hypercube
	def __call__(self, weights):
		return backend.clip(weights, -self.clip_value, self.clip_value)

	# get the config
	def get_config(self):
		return {'clip_value': self.clip_value}

# calculate wasserstein loss
def wasserstein_loss(y_true, y_pred):
	return backend.mean(y_true * y_pred)

# define the standalone critic model
def define_critic(in_shape=(28,28,1)):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# weight constraint
	const = ClipConstraint(0.01)
	# define model
	model = Sequential()
	# downsample to 14x14
	model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init, kernel_constraint=const, input_shape=in_shape))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))
	# downsample to 7x7
	model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init, kernel_constraint=const))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))
	# scoring, linear activation
	model.add(Flatten())
	model.add(Dense(1))
	# compile model
	opt = RMSprop(lr=0.00005)
	model.compile(loss=wasserstein_loss, optimizer=opt)
	return model

# define the standalone generator model
def define_generator(latent_dim):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# define model
	model = Sequential()
	# foundation for 7x7 image
	n_nodes = 128 * 7 * 7
	model.add(Dense(n_nodes, kernel_initializer=init, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((7, 7, 128)))
	# upsample to 14x14
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 28x28
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(BatchNormalization())
	model.add(LeakyReLU(alpha=0.2))
	# output 28x28x1
	model.add(Conv2D(1, (7,7), activation='tanh', padding='same', kernel_initializer=init))
	return model

# define the combined generator and critic model, for updating the generator
def define_gan(generator, critic):
	# make weights in the critic not trainable
	for layer in critic.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False
	# connect them
	model = Sequential()
	# add generator
	model.add(generator)
	# add the critic
	model.add(critic)
	# compile model
	opt = RMSprop(lr=0.00005)
	model.compile(loss=wasserstein_loss, optimizer=opt)
	return model

# load images
def load_real_samples():
	# load dataset
	(trainX, trainy), (_, _) = load_data()
	# select all of the examples for a given class
	selected_ix = trainy == 7
	X = trainX[selected_ix]
	# expand to 3d, e.g. add channels
	X = expand_dims(X, axis=-1)
	# convert from ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	return X

# select real samples
def generate_real_samples(dataset, n_samples):
	# choose random instances
	ix = randint(0, dataset.shape[0], n_samples)
	# select images
	X = dataset[ix]
	# generate class labels, -1 for 'real'
	y = -ones((n_samples, 1))
	return X, y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
	# generate points in the latent space
	x_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, latent_dim, n_samples):
	# generate points in latent space
	x_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	X = generator.predict(x_input)
	# create class labels with 1.0 for 'fake'
	y = ones((n_samples, 1))
	return X, y

# generate samples and save as a plot and save the model
def summarize_performance(step, g_model, latent_dim, n_samples=100):
	# prepare fake examples
	X, _ = generate_fake_samples(g_model, latent_dim, n_samples)
	# scale from [-1,1] to [0,1]
	X = (X + 1) / 2.0
	# plot images
	for i in range(10 * 10):
		# define subplot
		pyplot.subplot(10, 10, 1 + i)
		# turn off axis
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(X[i, :, :, 0], cmap='gray_r')
	# save plot to file
	filename1 = 'generated_plot_%04d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()
	# save the generator model
	filename2 = 'model_%04d.h5' % (step+1)
	g_model.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

# create a line plot of loss for the gan and save to file
def plot_history(d1_hist, d2_hist, g_hist):
	# plot history
	pyplot.plot(d1_hist, label='crit_real')
	pyplot.plot(d2_hist, label='crit_fake')
	pyplot.plot(g_hist, label='gen')
	pyplot.legend()
	pyplot.savefig('plot_line_plot_loss.png')
	pyplot.close()

# train the generator and critic
def train(g_model, c_model, gan_model, dataset, latent_dim, n_epochs=10, n_batch=64, n_critic=5):
	# calculate the number of batches per training epoch
	bat_per_epo = int(dataset.shape[0] / n_batch)
	# calculate the number of training iterations
	n_steps = bat_per_epo * n_epochs
	# calculate the size of half a batch of samples
	half_batch = int(n_batch / 2)
	# lists for keeping track of loss
	c1_hist, c2_hist, g_hist = list(), list(), list()
	# manually enumerate epochs
	for i in range(n_steps):
		# update the critic more than the generator
		c1_tmp, c2_tmp = list(), list()
		for _ in range(n_critic):
			# get randomly selected 'real' samples
			X_real, y_real = generate_real_samples(dataset, half_batch)
			# update critic model weights
			c_loss1 = c_model.train_on_batch(X_real, y_real)
			c1_tmp.append(c_loss1)
			# generate 'fake' examples
			X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
			# update critic model weights
			c_loss2 = c_model.train_on_batch(X_fake, y_fake)
			c2_tmp.append(c_loss2)
		# store critic loss
		c1_hist.append(mean(c1_tmp))
		c2_hist.append(mean(c2_tmp))
		# prepare points in latent space as input for the generator
		X_gan = generate_latent_points(latent_dim, n_batch)
		# create inverted labels for the fake samples
		y_gan = -ones((n_batch, 1))
		# update the generator via the critic's error
		g_loss = gan_model.train_on_batch(X_gan, y_gan)
		g_hist.append(g_loss)
		# summarize loss on this batch
		print('>%d, c1=%.3f, c2=%.3f g=%.3f' % (i+1, c1_hist[-1], c2_hist[-1], g_loss))
		# evaluate the model performance every 'epoch'
		if (i+1) % bat_per_epo == 0:
			summarize_performance(i, g_model, latent_dim)
	# line plots of loss
	plot_history(c1_hist, c2_hist, g_hist)

# size of the latent space
latent_dim = 50
# create the critic
critic = define_critic()
# create the generator
generator = define_generator(latent_dim)
# create the gan
gan_model = define_gan(generator, critic)
# load image data
dataset = load_real_samples()
print(dataset.shape)
# train model
train(generator, critic, gan_model, dataset, latent_dim)

---

➡️ **Next / 下一步**: File 2 of 2